# Anomaly Detection - Isolation Forest

Виявлення аномальних відгуків на основі engineered features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Завантаження даних

In [ ]:
df = pd.read_csv('../../data/doctors_reviews_engineered.csv')
print(f"Завантажено {len(df):,} відгуків")
print(f"Колонок: {len(df.columns)}")

## 2. Підготовка feature matrix

In [ ]:
# Вибираємо features для anomaly detection
feature_columns = [
    # Текстові ознаки
    'is_empty', 'text_length', 'word_count', 'avg_word_length', 
    'unique_words_ratio', 'exclamation_count', 'question_count', 
    'comma_count', 'sentence_count', 'uppercase_ratio', 'digit_count',
    
    # Behavioral ознаки
    'doctor_total_reviews', 'doctor_empty_ratio',
    
    # Anonymity ознаки
    'is_anonymous', 'no_reviewer_name', 'doctor_anonymous_ratio'
]

# Додаємо temporal та burst features якщо є
if 'reviews_on_day' in df.columns:
    feature_columns.extend(['reviews_on_day', 'doctor_max_daily_reviews'])
if 'day_of_week' in df.columns:
    feature_columns.extend(['day_of_week', 'month', 'is_weekend'])

# Фільтруємо тільки наявні колонки
available_features = [col for col in feature_columns if col in df.columns]
print(f"\nВикористовуємо {len(available_features)} ознак:")
for i, col in enumerate(available_features, 1):
    print(f"{i}. {col}")

In [ ]:
# Створюємо feature matrix
X = df[available_features].copy()

# Заповнюємо NaN значення медіаною/модою
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col].fillna(X[col].median(), inplace=True)
    else:
        X[col].fillna(X[col].mode()[0] if not X[col].mode().empty else 0, inplace=True)

print(f"\nFeature matrix: {X.shape}")
print(f"NaN values: {X.isnull().sum().sum()}")

In [ ]:
# Статистика features
print("\nСтатистика features:")
print(X.describe())

## 3. Нормалізація даних

In [ ]:
# Стандартизація features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean(axis=0)[:5]}...")
print(f"Std: {X_scaled.std(axis=0)[:5]}...")

## 4. Isolation Forest - базова модель

In [ ]:
# Тренуємо Isolation Forest
print("Тренування Isolation Forest...")
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,  # Очікуємо ~5% аномалій
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Predict: 1 = normal, -1 = anomaly
predictions = iso_forest.fit_predict(X_scaled)

# Anomaly scores (чим нижче - тим більш аномальний)
anomaly_scores = iso_forest.score_samples(X_scaled)

# Інвертуємо scores щоб вищий = більш аномальний
anomaly_scores_inverted = -anomaly_scores

print(f"\nПрогнози: {len(predictions):,}")
print(f"Нормальних: {(predictions == 1).sum():,} ({(predictions == 1).sum()/len(predictions)*100:.2f}%)")
print(f"Аномалій: {(predictions == -1).sum():,} ({(predictions == -1).sum()/len(predictions)*100:.2f}%)")

In [ ]:
# Додаємо результати до датасету
df['is_anomaly'] = (predictions == -1).astype(int)
df['anomaly_score'] = anomaly_scores_inverted

print("Anomaly detection results додано до датасету")

## 5. Аналіз anomaly scores

In [ ]:
# Розподіл anomaly scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram всіх scores
axes[0, 0].hist(anomaly_scores_inverted, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Anomaly Score')
axes[0, 0].set_ylabel('Частота')
axes[0, 0].set_title('Розподіл Anomaly Scores (всі)')
axes[0, 0].axvline(np.percentile(anomaly_scores_inverted, 95), 
                   color='red', linestyle='--', label='95-й перцентиль')
axes[0, 0].legend()

# Histogram окремо для normal vs anomaly
axes[0, 1].hist(anomaly_scores_inverted[predictions == 1], bins=50, 
                alpha=0.5, label='Normal', color='green')
axes[0, 1].hist(anomaly_scores_inverted[predictions == -1], bins=50, 
                alpha=0.5, label='Anomaly', color='red')
axes[0, 1].set_xlabel('Anomaly Score')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].set_title('Розподіл scores: Normal vs Anomaly')
axes[0, 1].legend()

# Boxplot
axes[1, 0].boxplot([anomaly_scores_inverted[predictions == 1], 
                     anomaly_scores_inverted[predictions == -1]],
                    labels=['Normal', 'Anomaly'],
                    vert=False)
axes[1, 0].set_xlabel('Anomaly Score')
axes[1, 0].set_title('Boxplot: Normal vs Anomaly')

# Cumulative distribution
sorted_scores = np.sort(anomaly_scores_inverted)
cumulative = np.arange(1, len(sorted_scores) + 1) / len(sorted_scores)
axes[1, 1].plot(sorted_scores, cumulative)
axes[1, 1].set_xlabel('Anomaly Score')
axes[1, 1].set_ylabel('Cumulative Probability')
axes[1, 1].set_title('Cumulative Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../data/anomaly_scores_analysis.png', dpi=150)
plt.show()

## 6. Характеристики аномальних відгуків

In [ ]:
# Порівнюємо характеристики normal vs anomaly
comparison = pd.DataFrame({
    'Normal': df[df['is_anomaly'] == 0][available_features].mean(),
    'Anomaly': df[df['is_anomaly'] == 1][available_features].mean()
}).round(2)

comparison['Різниця'] = (comparison['Anomaly'] - comparison['Normal']).round(2)
comparison['% зміни'] = ((comparison['Anomaly'] / comparison['Normal'] - 1) * 100).round(1)

print("\nПорівняння характеристик Normal vs Anomaly:")
print(comparison.sort_values('% зміни', ascending=False))

In [ ]:
# Візуалізація топ відмінностей
top_diffs = comparison['% зміни'].abs().nlargest(10)

plt.figure(figsize=(12, 6))
colors = ['red' if x < 0 else 'green' for x in comparison.loc[top_diffs.index, '% зміни']]
plt.barh(range(len(top_diffs)), comparison.loc[top_diffs.index, '% зміни'], color=colors)
plt.yticks(range(len(top_diffs)), top_diffs.index)
plt.xlabel('% зміни (Anomaly vs Normal)')
plt.title('Топ-10 відмінностей між Normal та Anomaly відгуками')
plt.axvline(0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('../../data/feature_differences_normal_vs_anomaly.png', dpi=150)
plt.show()

## 7. Топ аномальних відгуків

In [ ]:
# Топ-100 найбільш аномальних
top_anomalies = df.nlargest(100, 'anomaly_score')[[
    'Коментар', 'Ім\'я лікаря', 'Ім\'я коментатора', 'anomaly_score',
    'text_length', 'word_count', 'is_anonymous', 'doctor_total_reviews'
]]

print("\nТоп-20 найбільш аномальних відгуків:")
print(top_anomalies.head(20))

In [ ]:
# Деталі топ аномалій
print("\n" + "="*80)
print("ДЕТАЛІ ТОП-10 АНОМАЛЬНИХ ВІДГУКІВ")
print("="*80)

doctor_name_col = "Ім'я лікаря"
comentator_name_col = "Ім'я коментатора"

for i, (idx, row) in enumerate(top_anomalies.head(10).iterrows(), 1):
    print(f"\n{i}. Anomaly Score: {row['anomaly_score']:.4f}")
    print(f"   Лікар: {row[doctor_name_col]}")
    print(f"   Коментатор: {row[comentator_name_col]}")
    print(f"   Довжина: {row['text_length']} символів, {row['word_count']} слів")
    print(f"   Анонімний: {'Так' if row['is_anonymous'] else 'Ні'}")
    print(f"   Відгуків на лікаря: {row['doctor_total_reviews']}")
    print(f"   Текст: {row['Коментар']}")
    print("-" * 80)

## 8. Feature Importance аналіз

In [ ]:
# Для Isolation Forest немає прямого feature importance,
# але можемо оцінити через correlation з anomaly scores

feature_importance = pd.DataFrame({
    'feature': available_features,
    'correlation': [np.corrcoef(X[feat], anomaly_scores_inverted)[0, 1] 
                    for feat in available_features]
}).sort_values('correlation', key=abs, ascending=False)

print("\nFeature Importance:")
print(feature_importance)

In [ ]:
# Візуалізація feature importance
plt.figure(figsize=(10, 8))
colors = ['red' if x < 0 else 'green' for x in feature_importance['correlation']]
plt.barh(range(len(feature_importance)), feature_importance['correlation'], color=colors, alpha=0.7)
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Correlation з Anomaly Score')
plt.title('Feature Importance для Anomaly Detection')
plt.axvline(0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('../../data/feature_importance_anomaly.png', dpi=150)
plt.show()

## 9. PCA Візуалізація

In [ ]:
# PCA для візуалізації в 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance: {pca.explained_variance_ratio_}")
print(f"Total: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Візуалізація PCA з аномаліями
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Normal vs Anomaly
axes[0].scatter(X_pca[predictions == 1, 0], X_pca[predictions == 1, 1], 
                c='lightblue', alpha=0.3, s=10, label='Normal')
axes[0].scatter(X_pca[predictions == -1, 0], X_pca[predictions == -1, 1], 
                c='red', alpha=0.6, s=30, label='Anomaly')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('PCA: Normal vs Anomaly')
axes[0].legend()

# Колоруємо за anomaly score
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], 
                         c=anomaly_scores_inverted, cmap='RdYlGn_r', 
                         alpha=0.5, s=10)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('PCA: колоруємо за Anomaly Score')
plt.colorbar(scatter, ax=axes[1], label='Anomaly Score')

plt.tight_layout()
plt.savefig('../../data/pca_anomaly_visualization.png', dpi=150)
plt.show()

## 10. Збереження результатів

In [ ]:
# Зберігаємо датасет з anomaly detection results
df.to_csv('../../data/reviews_with_anomalies.csv', index=False)
print(f"Датасет з anomaly detection збережено: {len(df):,} записів")

# Зберігаємо топ аномалії окремо
top_anomalies.to_csv('../../data/top_anomalies_isolation_forest.csv', index=False)
print(f"Топ-100 аномалій збережено")